# LAB03: проверка промпта и базовой 2FA, 12.11.2026

Три синтетических запроса, новая демонстрационная БЗ. Данные и ожидаемые режимы зафиксированы до генераций. Этот ноутбук читает сохранённые ответы, биллинг и Langfuse; новых вызовов нет. Это диагностический повтор после ручного просмотра, не независимая accuracy.

In [1]:
import json,sys
from pathlib import Path
from collections import Counter
from decimal import Decimal
TASK=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ml/contracts/validation.py').is_file());RUN=TASK/'ml/experiments/cascade/evidence/prompt-followup-live-20261112'
sys.path[:0]=[str(TASK)]
from jsonschema import Draft202012Validator
from ml.contracts.validation import SCHEMA
from ml.models.baseline import retrieve
config=json.loads((RUN/'config.json').read_text(encoding='utf-8'))
data_paths=config.get('source_runs',[str(RUN.relative_to(TASK))])
data_paths=[str(Path('ml/experiments/cascade')/p) if p.startswith('evidence/') else p for p in data_paths]
rows=[]
for directory in data_paths:
    for line in (TASK/directory/'results.jsonl').read_text(encoding='utf-8').splitlines():
        row=json.loads(line);row['_evidence_run']=directory;rows.append(row)
summary=json.loads((RUN/'summary.json').read_text(encoding='utf-8'))
print('Запланировано:',len(config['cases']),'выполнено:',len(rows))
print(json.dumps(summary['groups'],ensure_ascii=False,indent=2))

Запланировано: 3 выполнено: 3
{
  "ordinary": {
    "jobs": 2,
    "expected_mode_matches": 1,
    "automatic": 1,
    "manual": 1,
    "mistral_http_200": 0,
    "mistral_gate_passed": 0,
    "sol_http_200": 2,
    "sol_attempts": 2,
    "reasons": {
      "complete_coverage": 1,
      "assessment_clarify_fields": 1
    },
    "required_articles": 3,
    "found_required_articles": 3,
    "confirmed_micro_usd": 16774,
    "unknown_sol_reserve_micro_usd": 0,
    "median_end_to_end_ms": 36922.0,
    "confirmed_cost_rub": "1.492886",
    "unknown_reserve_rub": "0"
  },
  "boundary": {
    "jobs": 1,
    "expected_mode_matches": 1,
    "automatic": 0,
    "manual": 1,
    "mistral_http_200": 0,
    "mistral_gate_passed": 0,
    "sol_http_200": 1,
    "sol_attempts": 1,
    "reasons": {
      "evidence_provenance": 1
    },
    "required_articles": 0,
    "found_required_articles": 0,
    "confirmed_micro_usd": 6457,
    "unknown_sol_reserve_micro_usd": 0,
    "median_end_to_end_ms": 28453,

In [2]:
kb=[json.loads(s) for s in (TASK/'ml/data/cascade/kb.jsonl').read_text(encoding='utf-8').splitlines()]
needed=found=0
for row in rows:
    q=row['input'];ids=[s['source_id'] for s in retrieve(q['ticket_text'],q['history'],q['kb_version'],kb)]
    assert ids==row['initial_ids']
    expected=set(row['expected']['required_sources']);needed+=len(expected);found+=len(expected&set(ids))
    print(row['case_id'],'ожидалось',sorted(expected),'первый поиск',ids)
print('Recall необходимых статей:',found,'/',needed)
print('Простые формулировки этого плана не заменяют старый составной тест на четыре вопроса.')

two_factor_steps ожидалось ['KB-2FA-demo-20261112'] первый поиск ['KB-2FA-demo-20261112']
two_factor_and_invoice ожидалось ['KB-2FA-demo-20261112', 'KB-INVOICE-demo-20261112'] первый поиск ['KB-INVOICE-demo-20261112', 'KB-2FA-demo-20261112']
two_factor_lost_phone ожидалось [] первый поиск ['KB-2FA-demo-20261112', 'KB-ERROR-demo-20261112']
Recall необходимых статей: 3 / 3
Простые формулировки этого плана не заменяют старый составной тест на четыре вопроса.


In [3]:
for row in rows:
    card=row['proposal']['card'];assert Draft202012Validator(SCHEMA).is_valid(card)
    stages=[(a['provider'],(a.get('diagnostics') or {}).get('route_reason')) for a in row['attempts']]
    print(row['case_id'],row['expected']['group'],stages,'итог:',card['mode'],row['job']['state'])
    print(card['draft_reply'] or card['summary'])
    if card['status']=='resolved':assert not card['needs_human_review']
print('Режим, корректность содержания, безопасность и доступность оцениваются отдельно.')

two_factor_steps ordinary [('mistral', 'attempt_deadline_exceeded'), ('sol', 'complete_coverage')] итог: answer ready
Базовые шаги для сервиса, поддерживающего 2FA через приложение-аутентификатор:
1. Войдите в аккаунт и откройте настройки безопасности.
2. Найдите настройку 2FA и выберите приложение-аутентификатор, если такой способ поддерживается.
3. Установите совместимое приложение-аутентификатор на телефон.
4. Добавьте аккаунт в приложении и отсканируйте QR-код, показанный самим сервисом.
5. Введите текущий одноразовый код из приложения только на странице настройки сервиса и завершите подтверждение по подсказкам.
6. Если сервис предложит резервные коды, сохраните их в защищённом месте.

Это учебный общий пример, а не проверенная инструкция интерфейса конкретного продукта; названия пунктов могут отличаться. Для точных пунктов используйте официальную инструкцию сервиса. Не отправляйте в поддержку QR-код, секрет настройки, одноразовые или резервные коды.
two_factor_and_invoice ordinary

In [4]:
billing={e['attempt_id']:e for directory in data_paths for p in (TASK/directory).glob('billing-*.json') for e in json.loads(p.read_text(encoding='utf-8'))['changes']}
total=Decimal(0)
for row in rows:
    for a in row['attempts']:
        entry=billing.get((a.get('diagnostics') or {}).get('attempt_id'))
        if entry:
            value=Decimal(entry['billed_micro_usd'])*89/1000000;total+=value
            print(row['case_id'],str(value)+' ₽',entry['provider_record']['id'])
before=summary['budget_before']['sol_accounted_micro_usd'];after=summary['budget_after']['sol_accounted_micro_usd']
print('Новые подтверждённые списания:',total,'₽')
print('Рост учёта, включая новые неизвестные резервы:',Decimal(after-before)*89/1000000,'₽')
assert Decimal(after-before)*89/1000000<=Decimal(10)
window_start=config['plan']['budget']['accounted_before_micro_usd']
assert Decimal(after-window_start)*89/1000000<=Decimal(10)
print('Курс 89 ₽/$; старые резервы не обнулены. Mistral бесплатна по условию аккаунта.')

two_factor_steps 0.758903 ₽ daf4d5e0-5cce-4a17-ae33-9d5b637e49ad
two_factor_and_invoice 0.733983 ₽ 54f9ad79-b023-4b4e-b5b9-9dc9767ccd23
two_factor_lost_phone 0.574673 ₽ fc0a1790-c4b7-4908-a62c-2577c9fbe8e6
Новые подтверждённые списания: 2.067559 ₽
Рост учёта, включая новые неизвестные резервы: 2.067559 ₽
Курс 89 ₽/$; старые резервы не обнулены. Mistral бесплатна по условию аккаунта.


In [5]:
expected_generations=0;received=0
for row in rows:
    calls=[a for a in row['attempts'] if not (a.get('diagnostics') or {}).get('not_sent',False)]
    expected_generations+=len(calls)
    if not calls:print(row['case_id'],'остановка до сети — generation не требуется');continue
    proof=json.loads((TASK/row['_evidence_run']/('langfuse-'+row['case_id']+'.json')).read_text(encoding='utf-8'))
    generations=[o for o in proof['observations'] if o['type']=='GENERATION']
    received+=len(generations)
    print(row['case_id'],[(o['name'],o.get('inputUsage'),o.get('outputUsage')) for o in generations])
    assert all(o.get('input') and (o.get('output') or o.get('level')=='ERROR') for o in generations)
print('Generation прочитано:',received,'ожидалось:',expected_generations)
assert received==expected_generations

two_factor_steps [('cascade-sol', 6656, 1005), ('cascade-mistral', 0, 0)]
two_factor_and_invoice [('cascade-sol', 6659, 934), ('cascade-mistral', 0, 0)]
two_factor_lost_phone [('cascade-sol', 6655, 485), ('cascade-mistral', 0, 0)]
Generation прочитано: 6 ожидалось: 6


[Карточки](../ml/experiments/cascade/review/prompt-followup-cards-20261112.html). [Отчёт и ограничения](../docs/experiments/lab3/prompt-followup-20261112.md). На момент генерации оценки были пусты; итоговый просмотр участника 3/3 сохранён в отчёте; старые оценки v3 не переносятся.